# Módulo 01 · Aula 4: Carregando documentos

**Duração:** 45 min

## Objetivos de aprendizagem

Ao final desta aula você será capaz de:

- explicar o que é um `Document` (`page_content` + `metadata`) e por que os **metadados** importam em RAG
  (filtrar a busca e citar a fonte);
- carregar os quatro formatos da base da TechNova: **Markdown** com front matter, **PDF** página a página com
  `pypdf`, **HTML** só com o conteúdo do artigo com `BeautifulSoup` e **CSV** linha a linha;
- inventariar a base inteira com `load_folder()` e uma tabela do pandas;
- reconhecer problemas reais de ingestão: PDF sem texto, HTML com lixo e encoding errado;
- usar a seção **1. Carregar** de `src/rag/ingestion.py`.

## Pré-requisitos

- Aulas 01.1 a 01.3: embeddings, busca por similaridade e o índice FAISS de `src/rag/vectorstore.py`.
- Python: `pathlib`, dicionários e pandas básico.

## O que vamos construir

Até aqui usamos a base de conhecimento já indexada. Nesta aula e na próxima abrimos a caixa da **ingestão**, o
caminho que leva os arquivos da pasta `data/documents/` até o índice. A ideia central:

> A qualidade do RAG começa na ingestão. Documento bem carregado, bem dividido e com bons metadados é metade da
> resposta certa.

Esta aula cuida da primeira etapa: transformar arquivos de formatos diferentes em uma lista uniforme de
`Document`.

```mermaid
flowchart LR
    MD[policies/*.md<br/>internal/*.md] --> L1[load_markdown]
    PDF[manuals/*.pdf] --> L2[load_pdf]
    HTML[help_center/*.html] --> L3[load_html]
    CSV[faq.csv] --> L4[load_csv]
    L1 & L2 & L3 & L4 --> D["list[Document]"]
    D -.aula 1.5.-> S[dividir]
    S -.aula 1.5.-> I[(índice)]
```

In [1]:
# Os notebooks rodam a partir da raiz do projeto: é lá que ficam o .env, a pasta data/ e o pacote src/.
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Chat e embeddings vêm do Azure OpenAI configurado no .env (veja src/config.py).
from src.config import settings

## 1. O `Document`: texto e metadados

Todo componente de RAG do LangChain (splitters, vector stores, retrievers) conversa pelo mesmo tipo:
`langchain_core.documents.Document`. Ele tem só dois campos:

- `page_content`: o **texto** que vira vetor e que o modelo vai ler;
- `metadata`: um dicionário com **tudo o que não é o texto**: de onde veio, página, seção, se está em vigor...

In [2]:
from langchain_core.documents import Document

doc = Document(
    page_content="Trocas podem ser solicitadas em até 30 dias corridos após a entrega do pedido.",
    metadata={
        "source": "policies/exchange.md",
        "type": "markdown",
        "title": "Política de Troca",
        "section": "Resumo da regra",
        "status": "active",
        "audience": "customers",
    },
)
print(doc.page_content)
print(doc.metadata)

Trocas podem ser solicitadas em até 30 dias corridos após a entrega do pedido.
{'source': 'policies/exchange.md', 'type': 'markdown', 'title': 'Política de Troca', 'section': 'Resumo da regra', 'status': 'active', 'audience': 'customers'}


Os metadados **não viram vetor**: o embedding é calculado só sobre `page_content`. Mesmo assim, eles fazem duas
coisas que o vetor sozinho não faz:

1. **Filtrar** a busca: nunca mostrar ao cliente uma política arquivada ou um procedimento interno, por mais
   parecido que o texto seja com a pergunta.
2. **Citar** a fonte: "Política de Troca, seção Resumo da regra" ou "Manual do Nova Buds, página 3".

Um exemplo sem índice nenhum, só com Python, usando trechos reais da base:

In [3]:
examples = [
    doc,
    Document(
        page_content="Frete grátis para compras acima de R$ 199,00, em todo o Brasil.",
        metadata={"source": "policies/shipping_2024.md", "title": "Política de Frete e Entrega",
                  "status": "archived", "audience": "customers"},
    ),
    Document(
        page_content="Reembolsos de R$ 5.000,01 a R$ 15.000,00 são aprovados pelo supervisor de atendimento.",
        metadata={"source": "internal/refund_approval_procedure.md", "title": "Procedimento Interno de Aprovação de Reembolsos",
                  "status": "active", "audience": "internal"},
    ),
]


def allowed_for_customers(document: Document) -> bool:
    """Só conteúdo em vigor e destinado a clientes pode chegar à resposta."""
    return document.metadata["status"] == "active" and document.metadata["audience"] == "customers"


def citation(document: Document) -> str:
    meta = document.metadata
    parts = [meta["title"], meta.get("section"), f"página {meta['page']}" if meta.get("page") else None]
    return " > ".join(p for p in parts if p) + f" ({meta['source']})"


for document in examples:
    status = "entra" if allowed_for_customers(document) else "BLOQUEADO"
    print(f"{status:<10} {citation(document)}")

entra      Política de Troca > Resumo da regra (policies/exchange.md)
BLOQUEADO  Política de Frete e Entrega (policies/shipping_2024.md)
BLOQUEADO  Procedimento Interno de Aprovação de Reembolsos (internal/refund_approval_procedure.md)


**Observe:** as três frases poderiam aparecer numa busca sobre frete ou reembolso. Só os metadados dizem que a
regra de R$ 199 é antiga e que o procedimento é interno. Se o carregador não gravar `status` e `audience`, nenhuma
etapa posterior consegue recuperar essa informação. É por isso que dizemos que a qualidade começa na ingestão.

Os metadados usados no projeto:

| Chave | Exemplo | Para que serve |
|---|---|---|
| `source` | `manuals/fn-001_nova_buds.pdf` | citação e atualização do índice (remover ou reindexar um arquivo) |
| `type` | `markdown`, `pdf`, `html`, `faq` | escolher como dividir; estatísticas |
| `title` | `Política de Troca` | citação e cabeçalho de contexto (aula 1.5) |
| `page` / `row` | `3` / `12` | citar a página do PDF ou a linha do CSV |
| `section`, `subsection` | `Resumo da regra` | citação e cabeçalho de contexto (aula 1.5) |
| `status` | `active`, `archived` | filtro: só documentos em vigor |
| `audience` | `customers`, `internal` | filtro: só o que o cliente pode ver |
| `updated_at` | `2026-06-15` | saber se o conteúdo está atualizado |

> 💡 **Dica:** um bom teste para decidir se algo é metadado: "vou precisar filtrar, ordenar ou citar por isso?".
> Se sim, grave no carregador. O filtro de verdade (`make_filter`) chega no módulo 02.

## 2. Markdown com front matter

As políticas e o procedimento interno estão em Markdown. No topo de cada arquivo há um **front matter**: um
bloco entre `---` com pares `chave: valor`. É a forma mais comum de guardar metadados junto do texto (sites
estáticos, wikis, bases de conhecimento).

Para virar `Document`, o front matter vira `metadata` e o resto vira `page_content`. Com `split("---", 2)` o texto
se divide em três partes: o que vem antes do primeiro `---` (vazio), o bloco de metadados e o corpo. Primeiro o
arquivo cru, depois o `Document`:

In [4]:
from pathlib import Path

DOCS = Path(settings.documents_dir)  # data/documents

exchange_raw = (DOCS / "policies/exchange.md").read_text(encoding="utf-8")
print("\n".join(exchange_raw.splitlines()[:12]))
print("...\n" + "=" * 100)

_, front_matter, body = exchange_raw.split("---", 2)
metadata = {}
for line in front_matter.strip().splitlines():
    key, value = line.split(":", 1)
    metadata[key.strip()] = value.strip()

exchange_doc = Document(
    page_content=body.strip(),
    metadata={**metadata, "source": "policies/exchange.md", "type": "markdown"},
)
print(exchange_doc.metadata)
print(f"\n{len(exchange_doc.page_content)} caracteres, começando por:\n{exchange_doc.page_content[:230]}...")

---
title: Política de Troca
category: policy
updated_at: 2026-06-15
version: 4
status: active
audience: customers
---

# Política de Troca

A TechNova é uma loja exclusivamente online e quer que você fique satisfeito com cada compra. Esta política explica quando e como trocar um produto comprado no site ou no app, quais condições precisam ser atendidas e quanto tempo cada etapa leva.
...
{'title': 'Política de Troca', 'category': 'policy', 'updated_at': '2026-06-15', 'version': '4', 'status': 'active', 'audience': 'customers', 'source': 'policies/exchange.md', 'type': 'markdown'}

5256 caracteres, começando por:
# Política de Troca

A TechNova é uma loja exclusivamente online e quer que você fique satisfeito com cada compra. Esta política explica quando e como trocar um produto comprado no site ou no app, quais condições precisam ser aten...


O front matter é o lugar onde a equipe de conteúdo marca o que está em vigor e para quem. Vamos ler o de todos
os arquivos Markdown da base:

O front matter é o lugar onde a equipe de conteúdo marca o que está em vigor e para quem. Vamos ler o de todos
os arquivos Markdown da base:

In [5]:
for path in sorted(DOCS.rglob("*.md")):
    _, front_matter, _ = path.read_text(encoding="utf-8").split("---", 2)
    fields = dict(line.split(": ", 1) for line in front_matter.strip().splitlines())
    print(f"{path.relative_to(DOCS).as_posix():<40} status={fields['status']:<9} "
          f"audience={fields['audience']:<10} updated_at={fields['updated_at']}")

internal/refund_approval_procedure.md    status=active    audience=internal   updated_at=2026-08-04
policies/cancellation.md                 status=active    audience=customers  updated_at=2026-07-01
policies/exchange.md                     status=active    audience=customers  updated_at=2026-06-15
policies/payment.md                      status=active    audience=customers  updated_at=2026-03-18
policies/privacy.md                      status=active    audience=customers  updated_at=2026-02-05
policies/refund.md                       status=active    audience=customers  updated_at=2026-07-01
policies/shipping.md                     status=active    audience=customers  updated_at=2026-04-10
policies/shipping_2024.md                status=archived  audience=customers  updated_at=2024-03-01
policies/warranty.md                     status=active    audience=customers  updated_at=2026-05-20


**Observe:** `policies/shipping_2024.md` está `archived` (é a regra antiga de frete grátis acima de R$ 199) e
`internal/refund_approval_procedure.md` é `internal`. Os dois continuam na base de propósito: arquivar em vez de
apagar mantém o histórico, e o filtro de metadados decide o que o cliente vê.

> ⚠️ O corpo do Markdown mantém as marcas `#`, `**` e `-`. Não removemos: elas ajudam o modelo a entender a
> estrutura e, na aula 1.5, os títulos `##` e `###` serão usados para dividir o texto por seção.

## 3. PDF página a página com `pypdf`

Os manuais de produtos estão em PDF. Um PDF não é um arquivo de texto: é um formato de **desenho de página**
(posições de letras, linhas, imagens), como mostram os primeiros bytes do arquivo. A biblioteca `pypdf` lê essa
estrutura e reconstrói o texto de cada página. O PDF também traz metadados próprios (título, assunto, autor), que
aproveitamos:

In [6]:
from pypdf import PdfReader

buds_path = DOCS / "manuals/fn-001_nova_buds.pdf"
print(buds_path.read_bytes()[:90], "\n")

reader = PdfReader(buds_path)
print("Páginas:", len(reader.pages))
for key, value in reader.metadata.items():
    print(f"  {key:<14} {value}")

print("\nTexto extraído da página 1:\n")
print(reader.pages[0].extract_text())

b'%PDF-1.3\n%\xe9\xeb\xf1\xbf\n1 0 obj\n<<\n/Count 5\n/Kids [3 0 R\n5 0 R\n7 0 R\n9 0 R\n11 0 R]\n/MediaBox [0 0 5' 

Páginas: 5
  /Author        TechNova
  /CreationDate  D:20260927230838Z
  /Creator       TechNova Documentação Técnica
  /Keywords      FN-001, Fone Nova Buds, manual, TechNova
  /Subject       SKU FN-001 | Fone Nova Buds
  /Title         Manual do usuário: Fone Nova Buds

Texto extraído da página 1:

TechNova
Fone Nova Buds
SKU: FN-001
Manual do usuário
Fone de ouvido intra-auricular sem fio com cancelamento de ruído ativo (ANC), modo ambiente,
conexão multiponto e configuração pelo app Nova Connect.
Leia este manual antes de usar o produto e guarde-o para consultas futuras. As políticas de troca, reembolso, garantia e
frete citadas aqui estão publicadas na íntegra na Central de Ajuda da TechNova.
Revisão 1.2, fevereiro de 2026
Página 1 de 5


Criamos **um `Document` por página**, com o número da página nos metadados. Assim a resposta pode citar
"Manual do Nova Buds, página 3", e o cliente encontra a informação no PDF.

In [7]:
pdf_docs = [
    Document(
        page_content=page.extract_text(),
        metadata={
            "source": buds_path.relative_to(DOCS).as_posix(),
            "type": "pdf",
            "title": reader.metadata.title,
            "product": reader.metadata.subject,
            "page": number,
        },
    )
    for number, page in enumerate(reader.pages, start=1)
]

for d in pdf_docs:
    first_lines = " | ".join(d.page_content.splitlines()[1:3])
    print(f"página {d.metadata['page']}: {len(d.page_content):>5} caracteres  {first_lines[:70]}")

# O texto de uma página por dentro: é aí que aparecem as limitações da extração.
print("\n" + "=" * 100)
page_3 = pdf_docs[2].page_content
print(page_3[:700])
print("...")
print(page_3[-160:])

página 1:   449 caracteres  Fone Nova Buds | SKU: FN-001
página 2:  2064 caracteres  1. Apresentação | Obrigado por escolher o Fone Nova Buds (SKU FN-001),
página 3:  2127 caracteres  4. Primeiros passos | 4.1 Primeira carga
página 4:  2563 caracteres  LED Significado | Pulsando lentamente com o cabo
página 5:  3186 caracteres  Problema Causa provável Solução | Só um fone toca Fone sem carga ou

TechNova | Fone Nova Buds (FN-001) | Manual do usuário
4. Primeiros passos
4.1 Primeira carga
Retire as películas de proteção dos contatos, coloque os fones no estojo e carregue até o LED ficar verde fixo
antes do primeiro uso.
4.2 Pareamento
- Abra o estojo com os fones dentro.
- Segure o botão do estojo por 3 segundos até o LED piscar branco. O modo de pareamento fica ativo por
3 minutos.
- No celular, ative o Bluetooth e selecione "Nova Buds" na lista de dispositivos.
- Quando a conexão for concluída, o LED fica branco fixo por 5 segundos e você ouve um sinal sonoro nos
fones.
4.3 Controles 

**Observe:**

- Toda página começa com o cabeçalho corrido `TechNova | Fone Nova Buds (FN-001) | Manual do usuário` e termina
  com `Página 3 de 5`. É ruído, mas ruído útil: o nome do produto no topo ajuda a busca (voltaremos a isso na
  aula 1.5).
- A tabela de controles por toque virou linhas de texto corrido ("Gesto Função... 1 toque..."). A extração não
  preserva o layout. Para manuais simples isso basta; para tabelas complexas existem extratores de layout, fora
  do escopo deste curso.
- As quebras de linha seguem a largura da página, não as frases.

## 4. HTML: só o `<article>`, sem menu e rodapé

Os artigos da central de ajuda são páginas HTML completas, copiadas do site. Veja o começo do arquivo cru:

In [8]:
track_path = DOCS / "help_center/track_order.html"
track_raw = track_path.read_text(encoding="utf-8")
print("\n".join(track_raw.splitlines()[:40]))
print("...")

<!doctype html>
<html lang="pt-BR">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Como rastrear seu pedido | Central de Ajuda TechNova</title>
  <meta name="description" content="Saiba onde encontrar o código de rastreio, como identificar a transportadora e o que significa cada status do seu pedido TechNova.">
  <meta name="updated_at" content="2026-08-12">
  <meta name="robots" content="index, follow">
  <link rel="stylesheet" href="/assets/css/central-ajuda.css">
  <script src="/assets/js/analytics.js" defer></script>
</head>
<body>
  <header class="topo">
    <a class="logo" href="/">TechNova</a>
    <form class="busca" action="/ajuda/busca" method="get">
      <label for="q">Buscar na Central de Ajuda</label>
      <input id="q" name="q" type="search" placeholder="Como podemos ajudar?">
      <button type="submit">Buscar</button>
    </form>
  </header>
  <nav class="menu-principal" aria-label="Menu principal">
    <

Tem `<head>`, busca, menu principal, trilha de navegação e, no fim do arquivo, rodapé. O conteúdo que interessa
está dentro de `<article>`. Com `BeautifulSoup` comparamos o texto da página inteira (`<body>`) com o do artigo:

In [9]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(track_raw, "html.parser")
body_text = soup.body.get_text("\n", strip=True)
article_text = soup.find("article").get_text("\n", strip=True)

print(f"<body>:    {len(body_text):>5} caracteres")
print(f"<article>: {len(article_text):>5} caracteres")

article_lines = set(article_text.splitlines())
junk = [line for line in body_text.splitlines() if line not in article_lines]
print(f"\n{len(junk)} linhas que só existem fora do artigo:")
print(junk)

# O mesmo rodapé se repete nos outros artigos?
footer_phrase = "Formas de pagamento: Pix, cartão de crédito e boleto bancário."
help_files = sorted((DOCS / "help_center").glob("*.html"))

with_phrase = [
    p.name for p in help_files
    if footer_phrase in BeautifulSoup(p.read_text(encoding="utf-8"), "html.parser").body.get_text()
]
print(f"\nA frase do rodapé aparece em {len(with_phrase)} de {len(help_files)} artigos.")

<body>:     4787 caracteres
<article>:  4063 caracteres

33 linhas que só existem fora do artigo:
['TechNova', 'Buscar na Central de Ajuda', 'Buscar', 'Início', 'Notebooks', 'Smartphones', 'Áudio', 'Monitores', 'Acessórios', 'Ofertas', 'Central de Ajuda', 'Minha Conta', 'Carrinho', 'Início', '>', 'Central de Ajuda', '>', 'Este artigo foi útil?', 'Sim', 'Não', 'Ainda precisa de ajuda?', 'Fale com a gente', 'Sobre a TechNova', 'Política de troca', 'Política de garantia', 'Política de frete', 'Política de privacidade', 'Fale conosco', 'Termos de uso', 'Formas de pagamento: Pix, cartão de crédito e boleto bancário.', 'Baixe o app TechNova e acompanhe seus pedidos de onde estiver.', 'TechNova Comércio Eletrônico S.A. | CNPJ 00.000.000/0001-00 | Loja exclusivamente online.', '© 2026 TechNova. Todos os direitos reservados. Preços e condições válidos apenas para compras no site e no app.']

A frase do rodapé aparece em 16 de 16 artigos.


Parece pouco (cerca de 15% do texto), mas o problema não é o tamanho: é a **repetição**. O mesmo menu e o
mesmo rodapé aparecem em **todos** os artigos. Se esse texto entrar no índice, uma pergunta como "quais as formas de pagamento?" encontra o **rodapé** de 16
artigos que não tratam de pagamento, e os links "Política de troca, Política de garantia..." aproximam todo
artigo de qualquer pergunta sobre políticas. Ruído repetido deixa os vetores parecidos entre si e atrapalha a
busca.

Montamos o `Document` só com o artigo, aproveitando `<title>` e as tags `<meta>` como metadados:

In [10]:
description = soup.find("meta", attrs={"name": "description"})["content"]
updated_at = soup.find("meta", attrs={"name": "updated_at"})["content"]

html_doc = Document(
    page_content=article_text,
    metadata={
        "source": track_path.relative_to(DOCS).as_posix(),
        "type": "html",
        "title": soup.title.get_text(strip=True),
        "description": description,
        "updated_at": updated_at,
    },
)
print(html_doc.metadata)
print()
print(html_doc.page_content[:400])

{'source': 'help_center/track_order.html', 'type': 'html', 'title': 'Como rastrear seu pedido | Central de Ajuda TechNova', 'description': 'Saiba onde encontrar o código de rastreio, como identificar a transportadora e o que significa cada status do seu pedido TechNova.', 'updated_at': '2026-08-12'}

Como rastrear seu pedido
Depois que o pagamento é confirmado, você pode acompanhar cada etapa do seu pedido pelo site ou pelo app TechNova. Neste artigo explicamos onde encontrar o código de rastreio, como saber qual transportadora está com a sua encomenda e o que fazer se o rastreio não mostrar novidades.
Onde encontrar o código de rastreio
O código de rastreio é gerado no momento em que o pedido


> 💡 **Dica:** cada site tem a sua estrutura. Antes de escrever o carregador, abra dois ou três arquivos e
> descubra qual elemento guarda o conteúdo (`<article>`, `<main>`, uma `div` específica). Escolher o seletor
> certo vale mais que qualquer limpeza posterior.

## 5. CSV: uma linha, um `Document`

As perguntas frequentes estão em `faq.csv`, com as colunas `question`, `answer` e `category`. Repare nas aspas:
a resposta da linha 3 tem vírgulas e até aspas dentro dela (`""Cancelar pedido""`). Por isso **nunca** se lê CSV
com `line.split(",")`:

In [11]:
faq_path = DOCS / "faq.csv"
print("\n".join(faq_path.read_text(encoding="utf-8").splitlines()[:4]))

line_3 = faq_path.read_text(encoding="utf-8").splitlines()[2]
print("\nsplit(',') na linha 3 devolve", len(line_3.split(",")), "campos (o arquivo tem 3 colunas)")

question,answer,category
Como acompanho o status do meu pedido?,"Acesse Minha Conta > Meus Pedidos. Lá aparecem o status (processando, em trânsito, entregue ou cancelado), a previsão de entrega e, depois do despacho, o código de rastreio.",orders
Posso cancelar meu pedido?,"Sim, sem custo, enquanto ele estiver com status processando, ou seja, antes do despacho. Use o botão ""Cancelar pedido"" em Minha Conta > Meus Pedidos ou fale com o atendimento. Depois do envio não é possível cancelar, mas você pode recusar a entrega ou pedir a devolução em até 30 dias após receber.",orders
Consigo mudar o endereço de um pedido já feito?,"Somente enquanto o pedido estiver processando, pelo chat com um atendente humano. Se o novo CEP mudar o frete ou o prazo, os valores são recalculados. Depois do despacho, o endereço não pode mais ser alterado.",orders

split(',') na linha 3 devolve 8 campos (o arquivo tem 3 colunas)


O módulo `csv` da biblioteca padrão trata aspas e vírgulas corretamente. Cada linha já é uma unidade de sentido
(uma pergunta com a sua resposta), então vira um `Document` inteiro, que nunca será dividido:

In [12]:
import csv

with faq_path.open(encoding="utf-8", newline="") as file:
    faq_docs = [
        Document(
            page_content=f"Pergunta: {row['question']}\nResposta: {row['answer']}",
            metadata={"source": "faq.csv", "type": "faq", "category": row["category"], "row": number},
        )
        for number, row in enumerate(csv.DictReader(file), start=2)  # a linha 1 é o cabeçalho
    ]

print(f"{len(faq_docs)} perguntas frequentes\n")
print(faq_docs[1].page_content)
print(faq_docs[1].metadata)

40 perguntas frequentes

Pergunta: Posso cancelar meu pedido?
Resposta: Sim, sem custo, enquanto ele estiver com status processando, ou seja, antes do despacho. Use o botão "Cancelar pedido" em Minha Conta > Meus Pedidos ou fale com o atendimento. Depois do envio não é possível cancelar, mas você pode recusar a entrega ou pedir a devolução em até 30 dias após receber.
{'source': 'faq.csv', 'type': 'faq', 'category': 'orders', 'row': 3}


**Observe:** o texto junta pergunta e resposta com rótulos. A pergunta ajuda a busca (clientes perguntam de jeitos
parecidos) e a resposta é o que o modelo vai usar. O número `row` é a linha do arquivo (começando em 2, porque a
linha 1 é o cabeçalho), útil para achar e corrigir uma FAQ.

Resumo dos quatro formatos:

| Formato | Pasta | Biblioteca | Unidade | O que fica de fora |
|---|---|---|---|---|
| Markdown | `policies/`, `internal/` | leitura de texto | 1 documento por arquivo | o bloco `---` (vira metadado) |
| PDF | `manuals/` | `pypdf` | 1 documento por página | layout e páginas sem texto |
| HTML | `help_center/` | `BeautifulSoup` | 1 documento por artigo | head, menu, trilha, rodapé, scripts |
| CSV | `faq.csv` | `csv` | 1 documento por linha | o cabeçalho (vira nomes de campos) |

## 6. Problemas reais de ingestão

A base da TechNova é bem comportada. Bases reais não são. Três problemas aparecem em quase todo projeto; vamos
simular cada um em uma pasta temporária (nada em `data/documents` é alterado).

### 6.1 PDF sem texto (digitalizado)

Um PDF escaneado é uma **imagem** de cada página: não há letras para extrair. Simulamos com um PDF de páginas em
branco, o mesmo resultado que o `pypdf` teria com uma digitalização:

In [13]:
import tempfile

from pypdf import PdfWriter

WORK_DIR = Path(tempfile.mkdtemp(prefix="technova_01_04_"))
print("Pasta temporária criada:", WORK_DIR.name)

writer = PdfWriter()
for _ in range(3):
    writer.add_blank_page(width=595, height=842)  # A4 em pontos
scanned_path = WORK_DIR / "scanned_manual.pdf"
with scanned_path.open("wb") as file:
    writer.write(file)

scanned = PdfReader(scanned_path)
print("Páginas:", len(scanned.pages))
print("Texto extraído por página:", [page.extract_text() for page in scanned.pages])

Pasta temporária criada: technova_01_04_thk2pktt
Páginas: 3
Texto extraído por página: ['', '', '']


Nenhum erro, nenhum aviso: só texto vazio. É o pior tipo de problema, porque passa em silêncio. O carregador do
projeto pula páginas sem texto (não faz sentido indexar um trecho vazio), então vale ter uma verificação que
**compare páginas com documentos**:

In [14]:
def pages_without_text(path: Path) -> list[int]:
    """Números das páginas de onde o pypdf não conseguiu extrair texto."""
    return [n for n, page in enumerate(PdfReader(path).pages, start=1) if not (page.extract_text() or "").strip()]


for path in [scanned_path, *sorted((DOCS / "manuals").glob("*.pdf"))]:
    empty = pages_without_text(path)
    print(f"{path.name:<28} {'OK' if not empty else f'SEM TEXTO nas páginas {empty}'}")

scanned_manual.pdf           SEM TEXTO nas páginas [1, 2, 3]
cx-001_nova_boom.pdf         OK
fn-001_nova_buds.pdf         OK
fn-002_nova_studio.pdf       OK
mn-001_nova_view_27.pdf      OK
nb-001_nova_air_14.pdf       OK
nb-002_nova_pro_16.pdf       OK


sm-001_nova_x.pdf            OK


sm-004_nova_mini.pdf         OK


A solução para PDFs digitalizados é **OCR** (reconhecimento óptico de caracteres, como o Tesseract) ou um serviço
de extração de documentos, antes da ingestão. O importante aqui é detectar e avisar, e não indexar páginas vazias
achando que está tudo certo.

### 6.2 HTML sem `<article>` e com lixo

Nem toda página tem um `<article>`. Uma estratégia robusta: procurar o artigo e, se não houver, usar o `<body>`,
removendo sempre os elementos que nunca são conteúdo (`script`, `style`, `nav`, `footer`). É o que o carregador
do projeto faz; vamos testar com uma página sem `<article>`:

In [15]:
from src.rag.ingestion import load_html

messy_html = """<html><head><title>Horário de atendimento</title><style>p {color: red}</style></head>
<body>
  <nav><a href="/">Início</a> <a href="/ofertas">Ofertas</a></nav>
  <h1>Horário de atendimento</h1>
  <p>O chat funciona de segunda a sábado, das 8h às 20h.</p>
  <script>trackPageView("atendimento");</script>
  <footer>© 2026 TechNova. Todos os direitos reservados.</footer>
</body></html>"""
messy_path = WORK_DIR / "no_article.html"
messy_path.write_text(messy_html, encoding="utf-8")

print("Texto bruto do <body>:")
print(BeautifulSoup(messy_html, "html.parser").body.get_text(" ", strip=True))
print("\nload_html():")
print(load_html(messy_path, WORK_DIR)[0].page_content)

Texto bruto do <body>:
Início Ofertas Horário de atendimento O chat funciona de segunda a sábado, das 8h às 20h. © 2026 TechNova. Todos os direitos reservados.

load_html():
Horário de atendimento

O chat funciona de segunda a sábado, das 8h às 20h.


### 6.3 Encoding

Todo arquivo de texto é uma sequência de bytes, e o **encoding** diz como transformar bytes em letras. A base
usa UTF-8, mas arquivos exportados de sistemas antigos (ou do Excel no Windows) costumam vir em Latin-1
(cp1252). Com acentos, isso quebra:

In [16]:
text = "---\ntitle: Política de Devolução\nstatus: active\n---\n\nA devolução é gratuita em até 7 dias."
latin1_path = WORK_DIR / "latin1_policy.md"
latin1_path.write_bytes(text.encode("cp1252"))

try:
    latin1_path.read_text(encoding="utf-8")
except UnicodeDecodeError as error:
    print("Lendo como UTF-8:", error)

# O erro contrário não levanta exceção: bytes UTF-8 lidos como Latin-1 viram "mojibake".
utf8_path = WORK_DIR / "utf8_policy.md"
utf8_path.write_text(text, encoding="utf-8")
print("\nUTF-8 lido como Latin-1:", utf8_path.read_text(encoding="latin-1").splitlines()[1])

# A correção: sabemos a origem (cp1252), então convertemos o arquivo para UTF-8 uma única vez.
latin1_path.write_text(latin1_path.read_bytes().decode("cp1252"), encoding="utf-8")
print("\nDepois de converter:", latin1_path.read_text(encoding="utf-8").splitlines()[1])

Lendo como UTF-8: 'utf-8' codec can't decode byte 0xed in position 14: invalid continuation byte

UTF-8 lido como Latin-1: title: PolÃ­tica de DevoluÃ§Ã£o

Depois de converter: title: Política de Devolução


Os dois casos são ruins, mas o segundo é pior: a palavra "DevoluÃ§Ã£o" entra no índice sem erro e
nunca mais é encontrada por quem busca "devolução". Regras práticas:

- declare o encoding sempre (`read_text(encoding="utf-8")`), nunca confie no padrão do sistema operacional;
- converta os arquivos para UTF-8 **uma vez**, na origem, em vez de adivinhar o encoding a cada leitura;
- procure sinais de mojibake (`Ã`, `Â`) no texto carregado.

## 7. E os document loaders do LangChain?

O LangChain tem dezenas de *document loaders* prontos (`PyPDFLoader`, `BSHTMLLoader`, `CSVLoader`,
`UnstructuredMarkdownLoader`...). Eles vivem no pacote `langchain-community`, que está **em descontinuação**: as
integrações estão migrando para pacotes dedicados e o pacote comunitário não é mais a via recomendada.

Por isso o projeto escreve os carregadores sobre as bibliotecas diretas, que são as mesmas que esses loaders
usavam por baixo:

| Loader do community | O que o projeto usa |
|---|---|
| `PyPDFLoader` | `pypdf.PdfReader` |
| `BSHTMLLoader` | `bs4.BeautifulSoup` |
| `CSVLoader` | `csv.DictReader` |
| `TextLoader` / `UnstructuredMarkdownLoader` | `Path.read_text` + `parse_front_matter` |

O custo é pequeno (cerca de 100 linhas) e o ganho é **controle**: escolhemos exatamente quais metadados cada
formato grava, com nomes padronizados (`source`, `page`, `status`...), e o que é descartado (menu, rodapé). A
única dependência do community que sobra no projeto é o FAISS, isolada em `src/rag/vectorstore.py`, como vimos
na aula 01.3.

## 8. Do notebook para o projeto: [src/rag/ingestion.py](../src/rag/ingestion.py)

A seção **1. Carregar** do arquivo reúne o que fizemos, com alguns cuidados a mais:

| Função | O que faz |
|---|---|
| `parse_front_matter(text)` | separa o bloco `---` do corpo do Markdown; tolera arquivos sem front matter |
| `load_markdown(path, base)` | 1 `Document` por arquivo, com o front matter como metadados |
| `load_pdf(path, base)` | 1 `Document` por página, com `title`, `product`, `page`, `total_pages`; pula páginas sem texto |
| `load_html(path, base)` | só o `<article>` (ou o `<body>`), sem `script`, `style`, `nav`, `footer` |
| `load_csv(path, base)` | 1 `Document` por linha, com `row` e `category` |
| `LOADERS` | dicionário extensão -> carregador |
| `load_file(path, base)` | escolhe o carregador pela extensão |
| `load_folder(folder)` | carrega a pasta inteira (e subpastas) em ordem alfabética |

Decisões de design:

- **`DEFAULT_METADATA`**: todo documento sai com `status="active"` e `audience="customers"`, a menos que o arquivo
  diga outra coisa. Assim o filtro do módulo 02 nunca encontra uma chave faltando.
- **`source` relativo à pasta base** (`policies/exchange.md`), com barras normais: o mesmo valor em qualquer
  sistema operacional e em qualquer máquina, o que é essencial para atualizar o índice (aula 1.5).
- **`_clean`** remove espaços repetidos e linhas em branco em excesso, comuns na extração de PDF e HTML.
- Todos os arquivos são lidos com `encoding="utf-8"` explícito.

Primeiro, um arquivo de cada formato com `load_file`, conferindo que o produto chega ao mesmo texto que montamos
à mão e que um formato desconhecido gera um erro claro:

In [17]:
from src.rag.ingestion import DEFAULT_METADATA, LOADERS, load_file, load_folder, parse_front_matter

print("Formatos suportados:", {ext: fn.__name__ for ext, fn in LOADERS.items()})
print("Metadados padrão:", DEFAULT_METADATA)
print()
for relative in ["policies/exchange.md", "manuals/fn-001_nova_buds.pdf", "help_center/track_order.html", "faq.csv"]:
    loaded = load_file(DOCS / relative, base=DOCS)
    print(f"{relative}: {len(loaded)} Document(s)")
    print("   ", loaded[0].metadata)

print()
product_md = load_file(DOCS / "policies/exchange.md", base=DOCS)[0]
product_pdf = load_file(buds_path, base=DOCS)
print("Markdown igual ao nosso:", product_md.page_content == exchange_doc.page_content)
print("Front matter igual ao nosso:", parse_front_matter(exchange_raw)[0] == metadata)
print("PDF: páginas", len(product_pdf), "| caracteres (nosso x produto):",
      sum(len(d.page_content) for d in pdf_docs), "x", sum(len(d.page_content) for d in product_pdf))

try:
    load_file(WORK_DIR / "contract.docx")
except ValueError as error:
    print("\nValueError:", error)

Formatos suportados: {'.md': 'load_markdown', '.pdf': 'load_pdf', '.html': 'load_html', '.csv': 'load_csv'}
Metadados padrão: {'status': 'active', 'audience': 'customers'}

policies/exchange.md: 1 Document(s)
    {'status': 'active', 'audience': 'customers', 'title': 'Política de Troca', 'category': 'policy', 'updated_at': '2026-06-15', 'version': '4', 'source': 'policies/exchange.md', 'type': 'markdown'}
manuals/fn-001_nova_buds.pdf: 5 Document(s)
    {'status': 'active', 'audience': 'customers', 'source': 'manuals/fn-001_nova_buds.pdf', 'type': 'pdf', 'category': 'manual', 'title': 'Manual do usuário: Fone Nova Buds', 'product': 'SKU FN-001 | Fone Nova Buds', 'page': 1, 'total_pages': 5}
help_center/track_order.html: 1 Document(s)
    {'status': 'active', 'audience': 'customers', 'source': 'help_center/track_order.html', 'type': 'html', 'category': 'help_center', 'title': 'Como rastrear seu pedido | Central de Ajuda TechNova', 'description': 'Saiba onde encontrar o código de rastreio

Markdown igual ao nosso: True
Front matter igual ao nosso: True
PDF: páginas 5 | caracteres (nosso x produto): 10389 x 10389

ValueError: Formato não suportado: .docx. Use .csv, .html, .md, .pdf.


Agora a base inteira com `load_folder()`. O log mostra quantos arquivos viraram quantos documentos:

In [18]:
import pandas as pd

documents = load_folder()  # padrão: settings.documents_dir

inventory = pd.DataFrame(
    [
        {
            "source": d.metadata["source"],
            "type": d.metadata["type"],
            "status": d.metadata["status"],
            "audience": d.metadata["audience"],
            "chars": len(d.page_content),
        }
        for d in documents
    ]
)
print(inventory["type"].value_counts().to_string())
inventory.head(8)

2026-09-27 21:02:05 | INFO     | src.rag.ingestion | Documentos carregados | arquivos=34 | documentos=105


type
faq         40
pdf         40
html        16
markdown     9


,source,type,status,audience,chars
0,faq.csv,faq,active,customers,232
1,faq.csv,faq,active,customers,345
2,faq.csv,faq,active,customers,277
3,faq.csv,faq,active,customers,283
4,faq.csv,faq,active,customers,269
5,faq.csv,faq,active,customers,218
6,faq.csv,faq,active,customers,233
7,faq.csv,faq,active,customers,266


Agrupando por tipo e por arquivo:

In [19]:
from IPython.display import display

by_type = inventory.groupby("type").agg(
    files=("source", "nunique"), documents=("source", "size"),
    min_chars=("chars", "min"), mean_chars=("chars", "mean"), max_chars=("chars", "max"),
)
display(by_type.round(0).astype(int))

by_file = (
    inventory.groupby(["source", "type", "status", "audience"], as_index=False)
    .agg(documents=("chars", "size"), chars=("chars", "sum"))
    .sort_values("chars", ascending=False)
)
print("\nFora do padrão (status ou público):")
print(by_file[(by_file["status"] != "active") | (by_file["audience"] != "customers")].to_string(index=False))
print(f"\n{len(by_file)} arquivos, {inventory['chars'].sum():,} caracteres no total".replace(",", "."))

,files,documents,min_chars,mean_chars,max_chars
type,,,,,
faq,1,40,132,234,345
html,16,16,3339,3689,4101
markdown,9,9,3490,4635,5411
pdf,8,40,449,1993,3186



Fora do padrão (status ou público):
                               source     type   status  audience  documents  chars
internal/refund_approval_procedure.md markdown   active  internal          1   5297
            policies/shipping_2024.md markdown archived customers          1   3490

34 arquivos. 189.805 caracteres no total


**Observe:**

- 34 arquivos viraram 105 documentos: 9 Markdown, 16 HTML, 40 páginas de PDF (8 manuais de 5 páginas) e 40
  linhas de FAQ.
- Os tamanhos variam muito: uma FAQ tem cerca de 230 caracteres; uma política, quase 5.000. Colocar tudo isso
  num vetor só por documento não é uma boa ideia, e é o assunto da próxima aula.
- Os dois documentos especiais aparecem com os metadados certos: é esse inventário que você deve olhar antes de
  indexar qualquer base nova.

Por último, a resiliência: um arquivo com problema não derruba a ingestão da pasta inteira. Criamos uma pasta com
um arquivo bom e outro em Latin-1 (o `load_folder` registra o erro no log, com o nome do arquivo, e segue):

In [20]:
mixed_dir = WORK_DIR / "mixed"
mixed_dir.mkdir()
(mixed_dir / "good.md").write_text("---\ntitle: Boa\n---\n\nTexto em UTF-8, com acentuação.", encoding="utf-8")
(mixed_dir / "bad.md").write_bytes("---\ntitle: Ruim\n---\n\nTexto em Latin-1, com acentuação.".encode("cp1252"))

loaded = load_folder(mixed_dir)
print("\nCarregados:", [d.metadata["source"] for d in loaded])

# Limpeza: removemos a pasta temporária desta aula.
import shutil

shutil.rmtree(WORK_DIR)
print("Pasta temporária removida:", not WORK_DIR.exists())

2026-09-27 21:02:05 | ERROR    | src.rag.ingestion | Falha ao carregar | arquivo=/private/tmp/claude-501/-Users-mitoura-Documents-LangGraph/27ed4402-274d-4728-9753-90c38a175204/scratchpad/rag_m2/tmp/technova_01_04_thk2pktt/mixed/bad.md
Traceback (most recent call last):
  File "/Users/mitoura/Documents/RAGLangChain/src/rag/ingestion.py", line 188, in load_folder
    documents.extend(load_file(file, root))
                     ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mitoura/Documents/RAGLangChain/src/rag/ingestion.py", line 178, in load_file
    return loader(path, Path(base) if base else path.parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mitoura/Documents/RAGLangChain/src/rag/ingestion.py", line 84, in load_markdown
    metadata, body = parse_front_matter(path.read_text(encoding="utf-8"))
                                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/pathlib.py", line 

2026-09-27 21:02:05 | INFO     | src.rag.ingestion | Documentos carregados | arquivos=2 | documentos=1



Carregados: ['good.md']
Pasta temporária removida: True


> ⚠️ Resiliência não é esconder erros. Em produção, some os arquivos que falharam e trate isso como alerta: um
> manual que não entrou no índice é uma pergunta que o assistente não vai saber responder.

## Resumo

- Um `Document` tem `page_content` (o texto, que vira vetor) e `metadata` (o resto). Metadados não entram no
  vetor, mas permitem **filtrar** (status, público) e **citar** (título, seção, página).
- Cada formato tem o seu carregador: Markdown com front matter vira metadados + corpo; PDF vira um documento por
  página com `pypdf`; HTML vira só o `<article>` com `BeautifulSoup`; CSV vira um documento por linha com `csv`.
- O que se descarta importa tanto quanto o que se carrega: menu e rodapé repetidos em todos os artigos
  aproximam vetores que deveriam ser diferentes.
- Problemas reais: PDF digitalizado (texto vazio, sem erro), HTML sem `<article>`, encoding errado (erro ou,
  pior, mojibake silencioso). Detecte, avise e corrija na origem.
- Os loaders do `langchain-community` estão em descontinuação; o projeto usa `pypdf`, `BeautifulSoup` e `csv`
  diretamente, com metadados padronizados.
- No produto: `load_file()` escolhe o carregador pela extensão (`LOADERS`) e `load_folder()` carrega a base
  inteira: 34 arquivos, 105 documentos.

## Próxima aula

Na aula 01.5 (`01_05_chunking_and_indexing`) vamos dividir esses 105 documentos em trechos (*chunking*), com o
`RecursiveCharacterTextSplitter`, o `MarkdownHeaderTextSplitter` e o cabeçalho de contexto, escolher o tamanho
dos trechos com um experimento e montar o índice com ids estáveis, que permitem atualizá-lo sem recriá-lo.